<a href="https://colab.research.google.com/github/mukeshv0112/DAAEXP1/blob/Mukesh/DAAEXP2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import time
import random
import string

def naive_search(text, pattern):
    n, m = len(text), len(pattern)
    matches, comparisons = [], 0
    for i in range(n - m + 1):
        j = 0
        while j < m:
            comparisons += 1
            if text[i + j] != pattern[j]:
                break
            j += 1
        if j == m:
            matches.append(i)
    return matches, comparisons

def compute_lps(pattern):
    m = len(pattern)
    lps = [0] * m
    length, i = 0, 1
    while i < m:
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        elif length != 0:
            length = lps[length - 1]
        else:
            lps[i] = 0
            i += 1
    return lps

def kmp_search(text, pattern):
    n, m = len(text), len(pattern)
    lps = compute_lps(pattern)
    matches, comparisons = [], 0
    i = j = 0
    while i < n:
        comparisons += 1
        if pattern[j] == text[i]:
            i += 1; j += 1
        if j == m:
            matches.append(i - j)
            j = lps[j - 1]
        elif i < n and pattern[j] != text[i]:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1
    return matches, comparisons

# Ex. No. 2 | Comparative Analysis of Naive, Rabin-Karp, and KMP Algorithms for String MatchingCS5303 – DAA Lab Chennai Institute of Technology | Dept. of CSE | Page 9
def rabin_karp(text, pattern, q=101):
    n, m = len(text), len(pattern)
    d = 256
    h = pow(d, m - 1, q)
    p_hash = t_hash = 0
    matches, comparisons = [], 0
    for i in range(m):
        p_hash = (d * p_hash + ord(pattern[i])) % q
        t_hash = (d * t_hash + ord(text[i])) % q
    for s in range(n - m + 1):
        if p_hash == t_hash:
            for k in range(m):
                comparisons += 1
                if text[s + k] != pattern[k]:
                    break
            else:
                matches.append(s)
        if s < n - m:
            t_hash = (d * (t_hash - ord(text[s]) * h) + ord(text[s + m])) % q
            if t_hash < 0:
                t_hash += q
    return matches, comparisons

# --- Main Execution ---
text = 'AABAACAADAABAABA'
pattern = 'AABA'
print(f'Text:    {text}')
print(f'Pattern: {pattern}')

m1, c1 = naive_search(text, pattern)
m2, c2 = kmp_search(text, pattern)
m3, c3 = rabin_karp(text, pattern)

print(f'\nNaive  -> Matches at: {m1}, Comparisons: {c1}')
print(f'KMP    -> Matches at: {m2}, Comparisons: {c2}')
print(f'RK     -> Matches at: {m3}, Comparisons: {c3}')

# Performance comparison
text_large = ''.join(random.choices('ABCD', k=10000))
patterns = ['AB', 'ABCD', 'ABCDAB', 'ABCDABCD']
print(f'\n{"Pattern":>12} {"Naive":>10} {"KMP":>10} {"RK":>10}')
print('-' * 50)
for p in patterns:
    _, c1 = naive_search(text_large, p)
    _, c2 = kmp_search(text_large, p)
    _, c3 = rabin_karp(text_large, p)
    print(f'{p:>12} {c1:>10} {c2:>10} {c3:>10}')

Text:    AABAACAADAABAABA
Pattern: AABA

Naive  -> Matches at: [0, 9, 12], Comparisons: 30
KMP    -> Matches at: [0, 9, 12], Comparisons: 18
RK     -> Matches at: [0, 9, 12], Comparisons: 12

     Pattern      Naive        KMP         RK
--------------------------------------------------
          AB      12485      10000       1272
        ABCD      13282      10000        245
      ABCDAB      13335      10014        166
    ABCDABCD      13333      10014        170


In [2]:
import heapq

# --- Union-Find for Kruskal ---
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank   = [0] * n

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # Path compression
        return self.parent[x]

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
        return True

def kruskal(n, edges):
    """edges: list of (weight, u, v)"""
    edges.sort()  # O(E log E)
    uf   = UnionFind(n)
    mst  = []
    cost = 0
    for w, u, v in edges:
        if uf.union(u, v):
            mst.append((u, v, w))
            cost += w
            if len(mst) == n - 1:
                break
    return mst, cost

def prim(n, adj, start=0):
    """adj: adjacency list {u: [(v, w), ...]}"""
    INF    = float('inf')
    key    = [INF] * n
    parent = [-1]  * n
    inMST  = [False] * n
    key[start] = 0
    pq = [(0, start)]
    mst = []
    cost = 0
    while pq:
        w, u = heapq.heappop(pq)
        if inMST[u]:
            continue
        inMST[u] = True
        if parent[u] != -1:
            mst.append((parent[u], u, w))
            cost += w
        for v, wt in adj.get(u, []):
            if not inMST[v] and wt < key[v]:
                key[v] = wt
                parent[v] = u
                heapq.heappush(pq, (wt, v))
    return mst, cost

# Ex. No. 3       Implementation of Kruskal's and Prim's Algorithms for Minimum Spanning Tree       CS5303 – DAA Lab Chennai Institute of Technology | Dept. of CSE | Page 13
# --- Graph Definition ---
n = 7
edges = [
    (7, 0, 1), (5, 0, 3), (8, 1, 2), (9, 1, 3),
    (7, 1, 4), (5, 2, 4), (15, 3, 4), (6, 3, 5),
    (8, 4, 5), (9, 4, 6), (11, 5, 6)
]
adj = {}
for w, u, v in edges:
    adj.setdefault(u, []).append((v, w))
    adj.setdefault(v, []).append((u, w))

k_mst, k_cost = kruskal(n, edges[:])
p_mst, p_cost = prim(n, adj)

print('=== Kruskal\'s MST ===')
for u, v, w in k_mst:
    print(f'  Edge ({u} - {v})  Weight: {w}')
print(f'  Total MST Cost: {k_cost}')

print('\n=== Prim\'s MST ===')
for u, v, w in p_mst:
    print(f'  Edge ({u} - {v})  Weight: {w}')
print(f'  Total MST Cost: {p_cost}')

=== Kruskal's MST ===
  Edge (0 - 3)  Weight: 5
  Edge (2 - 4)  Weight: 5
  Edge (3 - 5)  Weight: 6
  Edge (0 - 1)  Weight: 7
  Edge (1 - 4)  Weight: 7
  Edge (4 - 6)  Weight: 9
  Total MST Cost: 39

=== Prim's MST ===
  Edge (0 - 3)  Weight: 5
  Edge (3 - 5)  Weight: 6
  Edge (0 - 1)  Weight: 7
  Edge (1 - 4)  Weight: 7
  Edge (4 - 2)  Weight: 5
  Edge (4 - 6)  Weight: 9
  Total MST Cost: 39


In [4]:
def matrix_chain_order(dims):
    """
    Matrix Chain Multiplication using DP
    dims: list of dimensions, matrix i has dims[i-1] x dims[i]
    Time: O(n^3), Space: O(n^2)
    """
    n = len(dims) - 1
    # m[i][j] = minimum multiplications for matrices i..j
    m = [[0] * (n + 1) for _ in range(n + 1)]
    s = [[0] * (n + 1) for _ in range(n + 1)]

    # l is the chain length
    for l in range(2, n + 1):
        for i in range(1, n - l + 2):
            j = i + l - 1
            m[i][j] = float('inf')
            for k in range(i, j):
                cost = m[i][k] + m[k+1][j] + dims[i-1] * dims[k] * dims[j]
                if cost < m[i][j]:
                    m[i][j] = cost
                    s[i][j] = k
    return m, s

def print_optimal_parens(s, i, j):
    if i == j:
        return f'A{i}'
    k = s[i][j]
    left  = print_optimal_parens(s, i, k)
    right = print_optimal_parens(s, k + 1, j)
    return f'({left} x {right})'

def print_dp_table(m, n):
    print('\nDP Cost Table m[i][j]:')
    print(f'{"":>6}', end='')
    for j in range(1, n + 1):
        print(f'A{j:>8}', end='')
    print()
    for i in range(1, n + 1):
        print(f'A{i:<5}', end='')
        for j in range(1, n + 1):
            if j < i:
                print(f'{"---":>9}', end='')
            else:
                print(f'{m[i][j]:>9}', end='')
        print()

# A1(10x30), A2(30x5), A3(5x60), A4(60x10)
dims = [10, 30, 5, 60, 10]
n    = len(dims) - 1
print(f'Matrix Dimensions:')
for i in range(n):
    print(f'  A{i+1}: {dims[i]} x {dims[i+1]}')

m, s = matrix_chain_order(dims)
print(f'\nMinimum scalar multiplications: {m[1][n]}')
print(f'Optimal parenthesization: {print_optimal_parens(s, 1, n)}')
print_dp_table(m, n)

Matrix Dimensions:
  A1: 10 x 30
  A2: 30 x 5
  A3: 5 x 60
  A4: 60 x 10

Minimum scalar multiplications: 5000
Optimal parenthesization: ((A1 x A2) x (A3 x A4))

DP Cost Table m[i][j]:
      A       1A       2A       3A       4
A1            0     1500     4500     5000
A2          ---        0     9000     4500
A3          ---      ---        0     3000
A4          ---      ---      ---        0


In [6]:
def is_safe(board, row, col):
    for prev_row in range(row):
        placed = board[prev_row]
        # Ex. No. 7  Solving N-Queens Problem using Backtracking
        # CS5303 - DAA Lab Chennai Institute of Technology | Dept. of CSE | Page 25
        if placed == col:  # Same column
            return False
        if abs(prev_row - row) == abs(placed - col):  # Diagonal
            return False
    return True

def solve_n_queens(n):
    board = [-1] * n
    solutions = []
    backtrack_count = [0]

    def backtrack(row):
        if row == n:
            solutions.append(board[:])
            return
        for col in range(n):
            if is_safe(board, row, col):
                board[row] = col
                backtrack(row + 1)
                board[row] = -1  # Undo
                backtrack_count[0] += 1

    backtrack(0)
    return solutions, backtrack_count[0]

def display_board(solution, n):
    print('  +' + '---+' * n)
    for row in range(n):
        print('  |', end='')
        for col in range(n):
            if solution[row] == col:
                print(' Q |', end='')
            else:
                print(' . |', end='')
        print()
        print('  +' + '---+' * n)

# --- Solve for N=4 (show all) and N=8 (count only) ---
for n in [4, 6, 8]:
    solutions, backtracks = solve_n_queens(n)
    print(f'N={n}: {len(solutions)} solutions, {backtracks} backtracks')
    if n == 4:
        print(f'\n  All solutions for {n}-Queens:')
        for i, sol in enumerate(solutions, 1):
            print(f'\n  Solution {i}: {sol}')
            display_board(sol, n)


N=4: 2 solutions, 16 backtracks

  All solutions for 4-Queens:

  Solution 1: [1, 3, 0, 2]
  +---+---+---+---+
  | . | Q | . | . |
  +---+---+---+---+
  | . | . | . | Q |
  +---+---+---+---+
  | Q | . | . | . |
  +---+---+---+---+
  | . | . | Q | . |
  +---+---+---+---+

  Solution 2: [2, 0, 3, 1]
  +---+---+---+---+
  | . | . | Q | . |
  +---+---+---+---+
  | Q | . | . | . |
  +---+---+---+---+
  | . | . | . | Q |
  +---+---+---+---+
  | . | Q | . | . |
  +---+---+---+---+
N=6: 4 solutions, 152 backtracks
N=8: 92 solutions, 2056 backtracks


In [8]:
import heapq
from itertools import permutations

INF = float('inf')

def reduce_matrix(mat):
    """Reduce matrix and return reduction cost"""
    m = [row[:] for row in mat]
    n = len(m)
    cost = 0
    # Row reduction
    for i in range(n):
        row_min = min(m[i])
        if row_min is not None and row_min != INF:
            cost += row_min
            m[i] = [x - row_min if x != INF else INF for x in m[i]]
    # Column reduction
    for j in range(n):
        # Handle cases where column might contain only INF or be empty
        col_values = [m[i][j] for i in range(n) if m[i][j] != INF]
        if col_values:
            col_min = min(col_values)
        else:
            col_min = None

        if col_min is not None and col_min != INF:
            cost += col_min
            for i in range(n):
                if m[i][j] != INF:
                    m[i][j] -= col_min
    return m, cost

def tsp_brute_force(cost, n):
    """Brute force for verification"""
    cities = list(range(1, n))
    best_cost = INF
    best_path = None
    for perm in permutations(cities):
        path = [0] + list(perm) + [0]
        current_cost = 0
        is_valid_path = True
        for i in range(n):
            u, v = path[i], path[i+1]
            if cost[u][v] == INF:
                is_valid_path = False
                break
            current_cost += cost[u][v]

        if is_valid_path and current_cost < best_cost:
            best_cost = current_cost
            best_path = path
    return best_path, best_cost

# --- 5-city cost matrix ---
cost = [
    [INF,  10,   8,   9,   7],
    [ 10, INF,  10,   5,   6],
    [  8,  10, INF,   8,   9],
    [  9,   5,   8, INF,   6],
    [  7,   6,   9,   6, INF]
]
n = 5
cities = ['A', 'B', 'C', 'D', 'E']

best_path, best_cost = tsp_brute_force(cost, n)

print('5-City TSP - Cost Matrix:')
print(f'{"" :>4}', ' '.join(f'{c:>5}' for c in cities))
for i, row in enumerate(cost):
    r = ['INF' if x == INF else str(x) for x in row]
    print(f'{cities[i]:>4}', ' '.join(f'{v:>5}' for v in r))

# Ex. No. 8    Travelling Salesman Problem using Branch and Bound for Finding Optimal Path
# CS5303 - DAA Lab Chennai Institute of Technology | Dept. of CSE | Page 29

if best_path:
    print(f'\nOptimal Tour: {" -> ".join(cities[i] for i in best_path)}')
    print(f'Minimum Cost: {best_cost}')
    print(f'\nPath verification:')
    for i in range(n):
        u, v = best_path[i], best_path[i+1]
        print(f'  {cities[u]} -> {cities[v]}: cost = {cost[u][v]}')
else:
    print('\nNo valid tour found.')


5-City TSP - Cost Matrix:
         A     B     C     D     E
   A   INF    10     8     9     7
   B    10   INF    10     5     6
   C     8    10   INF     8     9
   D     9     5     8   INF     6
   E     7     6     9     6   INF

Optimal Tour: A -> C -> D -> B -> E -> A
Minimum Cost: 34

Path verification:
  A -> C: cost = 8
  C -> D: cost = 8
  D -> B: cost = 5
  B -> E: cost = 6
  E -> A: cost = 7


In [11]:
def first_fit(items, capacity=1.0):     bins = []  # Each bin stores remaining space     bin_contents = []     for item in items:         placed = False         for i, space in enumerate(bins):             if space >= item:                 bins[i] -= item                 bin_contents[i].append(item)                 placed = True                 break         if not placed:             bins.append(capacity - item)             bin_contents.append([item])     return bin_contents   def first_fit_decreasing(items, capacity=1.0):     return first_fit(sorted(items, reverse=True), capacity)   def best_fit_decreasing(items, capacity=1.0):     sorted_items = sorted(items, reverse=True)     bins = []     bin_contents = []     for item in sorted_items:         best_idx = -1         best_space = float('inf')         for i, space in enumerate(bins):             if space >= item and space - item < best_space:                 best_space = space - item                 best_idx = i         if best_idx >= 0:             bins[best_idx] -= item             bin_contents[best_idx].append(item)         else:             bins.append(capacity - item)             bin_contents.append([item])     return bin_contents   def display_bins(label, bins):     print(f'\n{label}: {len(bins)} bins')     for i, b in enumerate(bins, 1):         used = sum(b)         bar = '#' * int(used * 20)         print(f'  Bin {i}: {[round(x,1) for x in b]} | Used: {used:.1f} '","               f'[{bar:<20}]')   items = [0.5, 0.7, 0.3, 0.9, 0.2, 0.6, 0.8, 0.4, 0.1, 0.5] capacity = 1.0 lower_bound = -(-sum(items) // capacity)  # Ceiling division   print(f'Items: {items}') print(f'Capacity: {capacity}') print(f'Sum of items: {sum(items)}') print(f'Lower bound on bins: {int(lower_bound)}')   ff_bins  = first_fit(items) ffd_bins = first_fit_decreasing(items) bfd_bins = best_fit_decreasing(items)  Ex. No. 9                              Efficient Bin Packing using Approximation Algorithm                           CS5303 – DAA Lab Chennai Institute of Technology | Dept. of CSE | Page 33   display_bins('First Fit (FF)',           ff_bins) display_bins('First Fit Decreasing (FFD)', ffd_bins) display_bins('Best Fit Decreasing (BFD)', bfd_bins)   print(f'\nSummary: Lower Bound={int(lower_bound)}, FF={len(ff_bins)}, FFD={len(ffd_bins)}, BFD={len(bfd_bins)}')

In [13]:
import random
import time
import sys
sys.setrecursionlimit(20000)

comparisons = 0

def partition(arr, low, high):
    global comparisons
    pivot = arr[high]
    i = low - 1
    for j in range(low, high):
        comparisons += 1
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]
    arr[i + 1], arr[high] = arr[high], arr[i + 1]
    return i + 1

def deterministic_quicksort(arr, low, high):
    if low < high:
        pi = partition(arr, low, high)
        deterministic_quicksort(arr, low, pi - 1)
        deterministic_quicksort(arr, pi + 1, high)

def randomized_quicksort(arr, low, high):
    if low < high:
        # Randomize pivot
        rand_idx = random.randint(low, high)
        arr[rand_idx], arr[high] = arr[high], arr[rand_idx]
        pi = partition(arr, low, high)
        randomized_quicksort(arr, low, pi - 1)
        randomized_quicksort(arr, pi + 1, high)

def run_test(name, sort_fn, arr):
    global comparisons
    a = arr[:]
    comparisons = 0
    start = time.perf_counter()
    sort_fn(a, 0, len(a) - 1)
    elapsed = (time.perf_counter() - start) * 1000
    return comparisons, elapsed

N = 5000
test_cases = {
    'Random'      : [random.randint(1, 100000) for _ in range(N)],
    'Sorted'      : list(range(N)),
    'Reverse'     : list(range(N, 0, -1)),
    'Nearly Sorted': list(range(N))
}
# Make Nearly Sorted slightly shuffled
ns = test_cases['Nearly Sorted']
for _ in range(N // 20):
    i, j = random.randint(0, N-1), random.randint(0, N-1)
    ns[i], ns[j] = ns[j], ns[i]

print(f'{'Input Type':<16} {'DQS Comps':>12} {'DQS Time(ms)':>14} '
      f'{'RQS Comps':>12} {'RQS Time(ms)':>14}')

# Ex. No. 10                         Improving Quick Sort Efficiency using Randomized Algorithm
# CS5303 - DAA Lab Chennai Institute of Technology | Dept. of CSE | Page 37
print('-' * 72)
for case, arr in test_cases.items():
    d_comps, d_time = run_test('DQS', deterministic_quicksort, arr)
    r_comps, r_time = run_test('RQS', randomized_quicksort,    arr)
    print(f'{case:<16} {d_comps:>12} {d_time:>14.2f} {r_comps:>12} {r_time:>14.2f}')


Input Type          DQS Comps   DQS Time(ms)    RQS Comps   RQS Time(ms)
------------------------------------------------------------------------
Random                  72760          10.88        68710          12.55
Sorted               12497500        1787.61        68992          11.31
Reverse              12497500        1478.83        74140          11.89
Nearly Sorted          156303          19.60        66838          11.07


In [3]:
import heapq

def dijkstra(graph, source):
    """
    Dijkstra's Algorithm using Min-Heap
    Time: O((V + E) log V), Space: O(V)
    graph: dict {u: [(v, weight), ...]}, 0-indexed
    """
    n = len(graph)
    dist = [float('inf')] * n
    prev = [None] * n
    dist[source] = 0
    pq = [(0, source)]  # (distance, vertex)
    visited = set()

    while pq:
        d, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)
        for v, w in graph[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u
                heapq.heappush(pq, (dist[v], v))

    return dist, prev

def reconstruct_path(prev, source, target):
    path = []
    node = target
    while node is not None:
        path.append(node)
        node = prev[node]
    path.reverse()
    if path[0] == source:
        return path
    return []

# --- Graph Definition (Adjacency List) ---
graph = {
    0: [(1, 4), (2, 1)],
    1: [(3, 1)],
    2: [(1, 2), (3, 5)],
    3: [(4, 3)],
    4: [(5, 2)],
    5: []
}

source = 0
dist, prev = dijkstra(graph, source)

print(f'Shortest paths from vertex {source}:')
print(f'{"Vertex":>8} {"Distance":>10} {"Path":>30}')
print('-' * 55)
for v in range(len(graph)):
    path = reconstruct_path(prev, source, v)
    path_str = ' -> '.join(map(str, path)) if path else 'No path'
# Ex. No. 4                   Implementation of Single Source Shortest Path Algorithm (Dijkstra's)           CS5303 – DAA Lab Chennai Institute of Technology | Dept. of CSE | Page 17
    d = dist[v] if dist[v] != float('inf') else 'INF'
    print(f'{v:>8} {str(d):>10} {path_str:>30}')

Shortest paths from vertex 0:
  Vertex   Distance                           Path
-------------------------------------------------------
       0          0                              0
       1          3                    0 -> 2 -> 1
       2          1                         0 -> 2
       3          4               0 -> 2 -> 1 -> 3
       4          7          0 -> 2 -> 1 -> 3 -> 4
       5          9     0 -> 2 -> 1 -> 3 -> 4 -> 5


In [2]:
import random

comparison_count = 0  # Global counter

def min_max_dc(arr, low, high):
    global comparison_count
    # Base case: single element
    if low == high:
        return arr[low], arr[low]
    # Base case: two elements
    if high == low + 1:
        comparison_count += 1
        if arr[low] < arr[high]:
            return arr[low], arr[high]
        return arr[high], arr[low]

    # Divide
    mid = (low + high) // 2
    lmin, lmax = min_max_dc(arr, low, mid)
    rmin, rmax = min_max_dc(arr, mid + 1, high)

    # Conquer: combine with 2 comparisons
    comparison_count += 1
    overall_min = lmin if lmin < rmin else rmin
    comparison_count += 1
    overall_max = lmax if lmax > rmax else rmax
    return overall_min, overall_max

def min_max_naive(arr):
    mn, mx = arr[0], arr[0]
    comps = 0
    for x in arr[1:]:
        comps += 1
        if x < mn:
            mn = x
        comps += 1
        if x > mx:
            mx = x
    return mn, mx, comps

# --- Demonstration on small array ---
arr = [3, 1, 7, 4, 9, 2, 8, 5, 6, 0]
comparison_count = 0
mn, mx = min_max_dc(arr, 0, len(arr) - 1)
dc_comps = comparison_count
_, _, naive_comps = min_max_naive(arr)
print(f'Array: {arr}')
print(f'Min: {mn}, Max: {mx}')
print(f'D&C Comparisons: {dc_comps}')
print(f'Naive Comparisons: {naive_comps}')

# --- Performance Analysis ---
print(f'\n{"Size":>8} {"DC Comps":>12} {"Naive Comps":>14} {"Formula 3n/2-2":>16}')
print('-' * 56)
for size in [10, 100, 1000, 10000]:
    arr = [random.randint(1, 10000) for _ in range(size)]
    comparison_count = 0
    mn, mx = min_max_dc(arr, 0, len(arr) - 1)
    dc = comparison_count
    _, _, naive = min_max_naive(arr)
    # Ex. No. 5 |                  To Find Min-Max Value by Applying Divide and Conquer Technique             CS5303 – DAA Lab Chennai Institute of Technology | Dept. of CSE | Page 20
    formula = 3 * size // 2 - 2
    print(f'{size:>8} {dc:>12} {naive:>14} {formula:>16}')

Array: [3, 1, 7, 4, 9, 2, 8, 5, 6, 0]
Min: 0, Max: 9
D&C Comparisons: 14
Naive Comparisons: 18

    Size     DC Comps    Naive Comps   Formula 3n/2-2
--------------------------------------------------------
      10           14             18               13
     100          162            198              148
    1000         1510           1998             1498
   10000        15902          19998            14998
